# 04 Evaluation

**Owner:** Hayden. Only the owner edits and saves this notebook.

Scores matching and the agent on the same test set every time, so the baselines and the system can sit side by side.

- **Matching** is scored on the planted cases in `data/planted_cases.csv`: recall (the share of students who should match that the list contains) and near-misses that leaked in.
- **The agent** is scored on the 20 situations in `data/test_set.csv`: did it call the right tools in the right order (action correct), and did the whole step come out right (task success).

Checkpoint 1 runs the two baselines. Checkpoint 2 adds the real system in the same cells.

Save with File > Save a copy in GitHub after Runtime > Restart and run all.

In [ ]:
# Setup: get the latest repo files and load the data. Run this first, every session.
import os, subprocess, sys
REPO_URL = "https://github.com/haydenhollar/ai301-vol-gig.git"
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"], check=True)

sys.path.insert(0, f"{REPO}/src")
import importlib, volgig as vg
importlib.reload(vg)
import pandas as pd

data = vg.load_data(f"{REPO}/data")
RESULTS = f"{REPO}/results"
os.makedirs(RESULTS, exist_ok=True)
for k in ["students", "postings", "test_set", "planted"]:
    print(f"{k}: {data[k].shape}")

## Baseline 1: BM25 keyword search

The same pipeline as the real system, with BM25 in place of semantic search:

1. Hard filters in code: availability, travel radius, car, tools, license.
2. BM25 scores each student who passed, using their skill tags and extra-skills note. The query is the posting's description and category.
3. Every student with a score above zero goes on the list, meaning they share at least one word with the posting. BM25 has no similarity floor of its own, so any shared word counts.

BM25 only sees exact words. It should miss a student who describes the skill in other words, like S70, who "cuts the grass at church" for a lawn job, and it should let in a student who shares a word but not the skill.

In [ ]:
import re
from rank_bm25 import BM25Okapi

STOP = set("""a an and are as at be been but by can do for from had has have i i'm if in into is it its
just me my no not of on or our so that the their them there they this to up was we were will with you your
about after all also am any been both each every get got how more most need needs one only over some than
too very what when which who would looking""".split())

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

students = data["students"]
bm25 = BM25Okapi([tokens(vg.student_text(s)) for _, s in students.iterrows()])
row_of = {sid: i for i, sid in enumerate(students.index)}

def bm25_matches(posting_id):
    p = data["postings"].loc[posting_id]
    scores = bm25.get_scores(tokens(vg.posting_text(p)))
    passed = vg.hard_filter(posting_id, data)
    ranked = sorted(((scores[row_of[s]], s) for s in passed), reverse=True)
    return [s for score, s in ranked if score > 0]

bm25_rows = [vg.matching_scores(bm25_matches(r["posting_id"]), r) for _, r in data["planted"].iterrows()]
bm25_table = pd.DataFrame(bm25_rows)
bm25_table.to_csv(f"{RESULTS}/baseline_bm25_matching.csv", index=False)
bm25_table

## Baseline 2: fixed script

The fixed script does the same thing for every situation. It sends the match list to the poster, then invites every match. It never checks status, never books, and never answers a question.

The match list comes from Baseline 1 for now. At Checkpoint 2 the agent and the fixed script both use the real `find_matches`, so this score will be rerun on the same list.

The tools are mock versions in `src/volgig.py`. They log every call. Code refuses a call that breaks a rule, but the score counts what was attempted.

In [ ]:
def fixed_script(board, posting_id, matches):
    board.send_match_list(posting_id, matches)
    board.send_invite(posting_id, matches)
    return ""  # the fixed script never writes a reply

fixed_rows, fixed_logs = [], []
for _, sit in data["test_set"].iterrows():
    matches = bm25_matches(sit["posting_id"])
    board = vg.JobBoard(sit, matches, data)
    reply = fixed_script(board, sit["posting_id"], matches)
    fixed_rows.append(vg.score_situation(sit, board.log, reply, matches))
    for c in board.log:
        fixed_logs.append({"situation_id": sit["situation_id"], **c})

fixed_table = pd.DataFrame(fixed_rows)
fixed_table.to_csv(f"{RESULTS}/baseline_fixed_script.csv", index=False)
pd.DataFrame(fixed_logs).to_csv(f"{RESULTS}/baseline_fixed_script_log.csv", index=False)
fixed_table

## Baseline scores

In [ ]:
m = bm25_table.dropna(subset=["recall"])
summary = pd.DataFrame([
    {"baseline": "BM25 keyword search", "measure": "Matching recall, mean over planted postings",
     "score": round(m["recall"].mean(), 3), "n": len(m)},
    {"baseline": "BM25 keyword search", "measure": "Near-misses that leaked onto a list",
     "score": int(sum(len(x.split()) for x in bm25_table["near_misses_leaked"])), "n": len(bm25_table)},
    {"baseline": "Fixed script", "measure": "Action correct (share of 20 situations)",
     "score": round(fixed_table["action_correct"].mean(), 3), "n": len(fixed_table)},
    {"baseline": "Fixed script", "measure": "Task success (share of 20 situations)",
     "score": round(fixed_table["task_success"].mean(), 3), "n": len(fixed_table)},
    {"baseline": "Fixed script", "measure": "Situations where a forbidden student was contacted",
     "score": int((fixed_table["contacted_forbidden"] != "").sum()), "n": len(fixed_table)},
])
summary.to_csv(f"{RESULTS}/baseline_scores.csv", index=False)
display(summary)
print("Fixed script task success by difficulty:")
display(fixed_table.groupby("difficulty")["task_success"].agg(["mean", "count"]))